# scispaCy Multi-Model + Regex NER

This notebook runs multiple scispaCy biomedical NER models on the same input text and adds a regex layer for PHI, IDs, addresses, dates, medication instructions, labs, vitals, and clinical codes.

The output keeps the `source` column so you can see exactly which scispaCy model or regex pattern predicted each span.

## Install scispaCy Models

First install the root requirements:

```powershell
cd C:\Users\vasav\OneDrive\Documents\Deep-learning-Lab
python -m pip install -r requirements.txt
```

Then install the scispaCy model packages you want to use. The notebook will also print these commands if a model is missing.

```powershell
python -m pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
python -m pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
python -m pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz
python -m pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_craft_md-0.5.4.tar.gz
python -m pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_jnlpba_md-0.5.4.tar.gz
```

The core scispaCy model `en_core_sci_sm` detects broad biomedical mentions. The NER models detect typed entities from their training corpora.

## Model Coverage

| Model | Main labels / purpose |
| --- | --- |
| `en_core_sci_sm` | General biomedical mention detector. Usually emits generic `ENTITY` spans. |
| `en_ner_bc5cdr_md` | `DISEASE`, `CHEMICAL` |
| `en_ner_bionlp13cg_md` | Anatomy/pathology/organ/tissue/cell/cancer/chemical style labels |
| `en_ner_craft_md` | `GGP`, `SO`, `TAXON`, `CHEBI`, `GO`, `CL` |
| `en_ner_jnlpba_md` | `DNA`, `CELL_TYPE`, `CELL_LINE`, `RNA`, `PROTEIN` |

Regex adds deterministic extraction for PHI and structured clinical patterns that scispaCy models do not reliably cover.

In [32]:
from __future__ import annotations

import re
from dataclasses import dataclass

import pandas as pd
import scispacy
import spacy

pd.set_option("display.max_colwidth", 160)

MODEL_INSTALL_URLS = {
    "en_core_sci_sm": "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz",
    "en_ner_bc5cdr_md": "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz",
    "en_ner_bionlp13cg_md": "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz",
    "en_ner_craft_md": "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_craft_md-0.5.4.tar.gz",
    "en_ner_jnlpba_md": "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_jnlpba_md-0.5.4.tar.gz",
}

SCISPACY_MODEL_SPECS = [
    {"name": "en_core_sci_sm", "purpose": "general biomedical mentions"},
    {"name": "en_ner_bc5cdr_md", "purpose": "disease and chemical"},
    {"name": "en_ner_bionlp13cg_md", "purpose": "anatomy, pathology, cancer, tissue, cell, chemicals"},
    {"name": "en_ner_craft_md", "purpose": "genes, chemicals, taxonomy, ontology/cell labels"},
    {"name": "en_ner_jnlpba_md", "purpose": "DNA, RNA, protein, cell type, cell line"},
]

## Input Text

Edit `input_text` and rerun the notebook to test another note.

In [33]:
input_text = """
On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old male with severe knee pain and osteoarthritis.
MRN 123456, patient ID AB-9981, email john.jones@example.com, phone (555) 123-4567. He lives at 12 Main Street, Boston, MA 02118.
BP 140/90, HR 88, SpO2 97%, temperature 98.6 F, creatinine 1.4 mg/dL, HbA1c 7.2%, WBC 11.2 K/uL.
ICD-10 E11.9 and CPT 99213 were recorded. The patient has a family history of cancer and prior knee arthroscopy.
""".strip()

input_text

"On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old male with severe knee pain and osteoarthritis.\nMRN 123456, patient ID AB-9981, email john.jones@example.com, phone (555) 123-4567. He lives at 12 Main Street, Boston, MA 02118.\nBP 140/90, HR 88, SpO2 97%, temperature 98.6 F, creatinine 1.4 mg/dL, HbA1c 7.2%, WBC 11.2 K/uL.\nICD-10 E11.9 and CPT 99213 were recorded. The patient has a family history of cancer and prior knee arthroscopy."

## Load Available scispaCy Models

In [34]:
def load_available_scispacy_models(model_specs: list[dict]) -> dict[str, spacy.language.Language]:
    loaded = {}
    missing = []

    for spec in model_specs:
        name = spec["name"]
        try:
            loaded[name] = spacy.load(name)
            print(f"Loaded {name}: {spec['purpose']}")
        except OSError:
            missing.append(name)
            print(f"Missing {name}: {spec['purpose']}")

    if missing:
        print("\nInstall missing models with:")
        for name in missing:
            print(f"python -m pip install {MODEL_INSTALL_URLS[name]}")

    return loaded


loaded_models = load_available_scispacy_models(SCISPACY_MODEL_SPECS)
list(loaded_models)

C:\Users\vasav\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


Loaded en_core_sci_sm: general biomedical mentions
Loaded en_ner_bc5cdr_md: disease and chemical
Loaded en_ner_bionlp13cg_md: anatomy, pathology, cancer, tissue, cell, chemicals
Loaded en_ner_craft_md: genes, chemicals, taxonomy, ontology/cell labels
Loaded en_ner_jnlpba_md: DNA, RNA, protein, cell type, cell line


['en_core_sci_sm',
 'en_ner_bc5cdr_md',
 'en_ner_bionlp13cg_md',
 'en_ner_craft_md',
 'en_ner_jnlpba_md']

## Helper Functions

In [35]:
RESULT_COLUMNS = ["source", "method", "entity", "text", "start", "end", "score", "context"]


def context_window(text: str, start: int, end: int, window: int = 45) -> str:
    left = max(0, start - window)
    right = min(len(text), end + window)
    return text[left:right].replace("\n", " ")


def make_results_frame(rows: list[dict]) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame(columns=RESULT_COLUMNS)

    df = pd.DataFrame(rows, columns=RESULT_COLUMNS)
    df = df.drop_duplicates(subset=["source", "method", "entity", "text", "start", "end"])
    return df.sort_values(["start", "end", "method", "source", "entity"]).reset_index(drop=True)

## Run scispaCy Models

In [36]:
def run_scispacy_models(text: str, models: dict[str, spacy.language.Language]) -> pd.DataFrame:
    rows = []

    for model_name, nlp in models.items():
        doc = nlp(text)
        for ent in doc.ents:
            rows.append(
                {
                    "source": model_name,
                    "method": "scispacy",
                    "entity": ent.label_,
                    "text": ent.text,
                    "start": ent.start_char,
                    "end": ent.end_char,
                    "score": None,
                    "context": context_window(text, ent.start_char, ent.end_char),
                }
            )

    return make_results_frame(rows)


scispacy_entities = run_scispacy_models(input_text, loaded_models)
scispacy_entities

,source,method,entity,text,start,end,score,context
0,en_core_sci_sm,scispacy,ENTITY,Dr. Smith,14,23,None,On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen
1,en_core_sci_sm,scispacy,ENTITY,St. Mary's Hospital,27,46,None,On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily f
2,en_core_sci_sm,scispacy,ENTITY,ibuprofen,58,67,None,Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John
3,en_ner_bc5cdr_md,scispacy,CHEMICAL,ibuprofen,58,67,None,Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John
4,en_ner_bionlp13cg_md,scispacy,SIMPLE_CHEMICAL,ibuprofen,58,67,None,Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John
5,en_core_sci_sm,scispacy,ENTITY,PO,75,77,None,"Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a"
6,en_core_sci_sm,scispacy,ENTITY,days,96,100,None,"cribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old male with s"
7,en_core_sci_sm,scispacy,ENTITY,Mr. John Jones,104,118,None,"buprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old male with severe knee pain an"
8,en_ner_bionlp13cg_md,scispacy,ORGANISM,John Jones,108,118,None,"ofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old male with severe knee pain an"
9,en_core_sci_sm,scispacy,ENTITY,male,134,138,None,"for 7 days to Mr. John Jones, a 64-year-old male with severe knee pain and osteoarthritis. MR"


## Regex Pattern Layer

These patterns catch structured clinical/PHI spans that model-based biomedical NER often misses. They are deterministic, so review false positives before using them in a production pipeline.

In [37]:
@dataclass(frozen=True)
class RegexPattern:
    entity: str
    pattern: str
    flags: int = re.IGNORECASE
    group: int = 0


REGEX_PATTERNS = [
    RegexPattern("EMAIL", r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b"),
    RegexPattern("PHONE", r"(?<!\d)(?:\+?1[-.\s]?)?(?:\(\d{3}\)|\d{3})[-.\s]?\d{3}[-.\s]?\d{4}(?!\d)"),
    RegexPattern("SSN", r"\b\d{3}-\d{2}-\d{4}\b"),
    RegexPattern("URL", r"\bhttps?://[^\s)]+"),
    RegexPattern("IP_ADDRESS", r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
    RegexPattern("DATE", r"\b(?:0?[1-9]|1[0-2])[/-](?:0?[1-9]|[12]\d|3[01])[/-](?:19|20)\d{2}\b"),
    RegexPattern("DATE", r"\b(?:19|20)\d{2}[-/](?:0?[1-9]|1[0-2])[-/](?:0?[1-9]|[12]\d|3[01])\b"),
    RegexPattern("DATE", r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},?\s+(?:19|20)\d{2}\b"),
    RegexPattern("DATE_OF_BIRTH", r"\b(?:DOB|date of birth)\s*[:#-]?\s*(?:\d{1,2}[/-]\d{1,2}[/-](?:19|20)\d{2}|(?:19|20)\d{2}[-/]\d{1,2}[-/]\d{1,2})\b"),
    RegexPattern("TIME", r"\b\d{1,2}:\d{2}\s*(?:AM|PM|am|pm)?\b"),
    RegexPattern("AGE", r"\b(?:age(?:d)?\s*)?\d{1,3}\s*(?:-?\s*years? old|-?year-old|yo|y/o)\b|\bage\s*[:=]?\s*\d{1,3}\b"),
    RegexPattern("DOCTOR_NAME", r"\b(?:Dr|Doctor|Prof|Professor)\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2}\b", 0),
    RegexPattern("PATIENT_NAME", r"\b(?:Mr|Mrs|Ms|Miss)\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2}\b", 0),
    RegexPattern("PATIENT_NAME", r"\b(?:patient|pt|name)\s*(?:is|name|:)\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2})\b", re.IGNORECASE, 1),
    RegexPattern("HOSPITAL", r"\b((?:St\.?\s+)?[A-Z][A-Za-z'.-]+(?:\s+[A-Z][A-Za-z'.-]+){0,4}\s+(?:Hospital|Clinic|Medical Center|Health System|Healthcare|University Hospital|Cancer Center))\b", 0, 1),
    RegexPattern("ADDRESS", r"(?<![/.-])\b\d{1,6}\s+(?!(?:Dr|Doctor|Mr|Mrs|Ms|Prof)\.?\b)[A-Z0-9][A-Za-z0-9'.-]*(?:\s+[A-Za-z0-9'.-]+){0,5}\s+(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Drive|Dr|Lane|Ln|Court|Ct|Way|Place|Pl)\.?\b(?:,\s*[A-Z][A-Za-z .'-]+)?(?:,\s*[A-Z]{2})?(?:\s+\d{5}(?:-\d{4})?)?"),
    RegexPattern("ZIP_CODE", r"(?<!CPT\s)\b\d{5}(?:-\d{4})?\b"),
    RegexPattern("CITY_STATE_ZIP", r"\b[A-Z][A-Za-z .'-]+,\s*[A-Z]{2}\s+\d{5}(?:-\d{4})?\b"),
    RegexPattern("MRN", r"\b(?:MRN|M\.?R\.?N\.?|medical record(?: number)?|record number)\s*[:#-]?\s*[A-Z0-9-]{4,}\b"),
    RegexPattern("PATIENT_ID", r"\b(?:patient|member|account|visit|encounter)\s*(?:id|number|no\.?)\s*[:#-]?\s*[A-Z0-9-]{3,}\b"),
    RegexPattern("NPI", r"\bNPI\s*[:#-]?\s*\d{10}\b"),
    RegexPattern("ICD10_CODE", r"\b[A-TV-Z][0-9][0-9AB](?:\.[0-9A-TV-Z]{1,4})?\b"),
    RegexPattern("CPT_CODE", r"\bCPT\s*[:#-]?\s*\d{5}\b"),
    RegexPattern("DOSAGE", r"\b\d+(?:\.\d+)?\s*(?:mg|mcg|g|kg|mL|ml|L|units?|IU|%)\b"),
    RegexPattern("FREQUENCY", r"\b(?:once|twice|three times|four times)\s+(?:daily|a day|per day|weekly|monthly)\b|\b(?:q\d{1,2}h|qhs|qid|tid|bid|prn|daily|weekly|monthly)\b"),
    RegexPattern("DURATION", r"\bfor\s+\d+\s+(?:hours?|days?|weeks?|months?|years?)\b"),
    RegexPattern("ROUTE", r"\b(?:PO|IV|IM|SC|SQ|subcutaneous|intravenous|intramuscular|oral|topical|inhaled|nasal|rectal)\b"),
    RegexPattern("MEDICATION_CONTEXT", r"\b(?:prescribed|started|taking|takes|given|administered)\s+([A-Za-z][A-Za-z-]{2,})\b", re.IGNORECASE, 1),
    RegexPattern("BLOOD_PRESSURE", r"\b(?:BP|blood pressure)\s*[:=]?\s*\d{2,3}/\d{2,3}\b"),
    RegexPattern("HEART_RATE", r"\b(?:HR|heart rate|pulse)\s*[:=]?\s*\d{2,3}\s*(?:bpm)?\b"),
    RegexPattern("TEMPERATURE", r"\b(?:temp(?:erature)?\s*[:=]?\s*)?\d{2,3}(?:\.\d)?\s*(?:F|C|deg F|deg C)\b"),
    RegexPattern("OXYGEN_SATURATION", r"\b(?:SpO2|O2 sat|oxygen saturation)\s*[:=]?\s*\d{2,3}%(?=\W|$)"),
    RegexPattern("LAB_VALUE", r"\b(?:WBC|RBC|Hgb|Hb|hemoglobin|platelets?|creatinine|glucose|sodium|potassium|HbA1c|A1c|ALT|AST|BUN|INR|CRP)\s*[:=]?\s*\d+(?:\.\d+)?(?:\s*(?:mg/dL|g/dL|mmol/L|mEq/L|K/uL|%|U/L|ng/mL))?\b"),
    RegexPattern("FAMILY_HISTORY", r"\bfamily history of\s+([A-Za-z][A-Za-z\s-]{2,40}?)(?=\.|,|;|\sand\s(?:prior|past|previous)|$)", re.IGNORECASE, 1),
]

In [38]:
def trim_span(text: str, start: int, end: int) -> tuple[str, int, int]:
    raw_span = text[start:end]
    stripped_span = raw_span.strip(" ,.;:\n\t")
    leading_trim = len(raw_span) - len(raw_span.lstrip(" ,.;:\n\t"))
    new_start = start + leading_trim
    new_end = new_start + len(stripped_span)
    return stripped_span, new_start, new_end


def run_regex_patterns(text: str, patterns: list[RegexPattern]) -> pd.DataFrame:
    rows = []

    for pattern_spec in patterns:
        compiled_pattern = re.compile(pattern_spec.pattern, pattern_spec.flags)
        for match in compiled_pattern.finditer(text):
            start, end = match.span(pattern_spec.group)
            if start < 0 or end < 0:
                continue

            span_text, start, end = trim_span(text, start, end)
            if not span_text:
                continue

            rows.append(
                {
                    "source": f"regex:{pattern_spec.entity}",
                    "method": "regex",
                    "entity": pattern_spec.entity,
                    "text": span_text,
                    "start": start,
                    "end": end,
                    "score": None,
                    "context": context_window(text, start, end),
                }
            )

    return make_results_frame(rows)


regex_entities = run_regex_patterns(input_text, REGEX_PATTERNS)
regex_entities

,source,method,entity,text,start,end,score,context
0,regex:DATE,regex,DATE,03/12/2019,3,13,None,On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed
1,regex:DOCTOR_NAME,regex,DOCTOR_NAME,Dr. Smith,14,23,None,On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen
2,regex:HOSPITAL,regex,HOSPITAL,St. Mary's Hospital,27,46,None,On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily f
3,regex:MEDICATION_CONTEXT,regex,MEDICATION_CONTEXT,ibuprofen,58,67,None,Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John
4,regex:DOSAGE,regex,DOSAGE,400 mg,68,74,None,"at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones,"
5,regex:ROUTE,regex,ROUTE,PO,75,77,None,"Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a"
6,regex:FREQUENCY,regex,FREQUENCY,twice daily,78,89,None,"ry's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old"
7,regex:DURATION,regex,DURATION,for 7 days,90,100,None,"l prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old male with s"
8,regex:PATIENT_NAME,regex,PATIENT_NAME,Mr. John Jones,104,118,None,"buprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old male with severe knee pain an"
9,regex:AGE,regex,AGE,64-year-old,122,133,None,"twice daily for 7 days to Mr. John Jones, a 64-year-old male with severe knee pain and osteoarthriti"


## Combined Results By Source

This table shows every prediction from every loaded scispaCy model and every regex pattern.

In [39]:
all_entities = make_results_frame(
    pd.concat([scispacy_entities, regex_entities], ignore_index=True).to_dict("records")
)

all_entities

,source,method,entity,text,start,end,score,context
0,regex:DATE,regex,DATE,03/12/2019,3,13,None,On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed
1,regex:DOCTOR_NAME,regex,DOCTOR_NAME,Dr. Smith,14,23,None,On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen
2,en_core_sci_sm,scispacy,ENTITY,Dr. Smith,14,23,None,On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen
3,regex:HOSPITAL,regex,HOSPITAL,St. Mary's Hospital,27,46,None,On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily f
4,en_core_sci_sm,scispacy,ENTITY,St. Mary's Hospital,27,46,None,On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily f
...,...,...,...,...,...,...,...,...
80,en_ner_bc5cdr_md,scispacy,DISEASE,cancer,486,492,None,ecorded. The patient has a family history of cancer and prior knee arthroscopy.
81,en_ner_bionlp13cg_md,scispacy,CANCER,cancer,486,492,None,ecorded. The patient has a family history of cancer and prior knee arthroscopy.
82,en_core_sci_sm,scispacy,ENTITY,prior,497,502,None,e patient has a family history of cancer and prior knee arthroscopy.
83,en_ner_bionlp13cg_md,scispacy,TISSUE,knee,503,507,None,ent has a family history of cancer and prior knee arthroscopy.


## Combined Span Summary

This groups overlapping exact spans so you can see which models or regex patterns agreed on the same text.

In [40]:
if all_entities.empty:
    span_summary = pd.DataFrame(columns=["text", "start", "end", "entities", "sources", "methods", "context"])
else:
    span_summary = (
        all_entities.groupby(["start", "end", "text"], as_index=False)
        .agg(
            entities=("entity", lambda values: ", ".join(sorted(set(values)))),
            sources=("source", lambda values: ", ".join(sorted(set(values)))),
            methods=("method", lambda values: ", ".join(sorted(set(values)))),
            context=("context", "first"),
        )
        .sort_values(["start", "end"])
        .reset_index(drop=True)
    )

span_summary

,start,end,text,entities,sources,methods,context
0,3,13,03/12/2019,DATE,regex:DATE,regex,On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed
1,14,23,Dr. Smith,"DOCTOR_NAME, ENTITY","en_core_sci_sm, regex:DOCTOR_NAME","regex, scispacy",On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen
2,27,46,St. Mary's Hospital,"ENTITY, HOSPITAL","en_core_sci_sm, regex:HOSPITAL","regex, scispacy",On 03/12/2019 Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily f
3,58,67,ibuprofen,"CHEMICAL, ENTITY, MEDICATION_CONTEXT, SIMPLE_CHEMICAL","en_core_sci_sm, en_ner_bc5cdr_md, en_ner_bionlp13cg_md, regex:MEDICATION_CONTEXT","regex, scispacy",Dr. Smith at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John
4,68,74,400 mg,DOSAGE,regex:DOSAGE,regex,"at St. Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones,"
5,75,77,PO,"ENTITY, ROUTE","en_core_sci_sm, regex:ROUTE","regex, scispacy","Mary's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a"
6,78,89,twice daily,FREQUENCY,regex:FREQUENCY,regex,"ry's Hospital prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old"
7,90,100,for 7 days,DURATION,regex:DURATION,regex,"l prescribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old male with s"
8,96,100,days,ENTITY,en_core_sci_sm,scispacy,"cribed ibuprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old male with s"
9,104,118,Mr. John Jones,"ENTITY, PATIENT_NAME","en_core_sci_sm, regex:PATIENT_NAME","regex, scispacy","buprofen 400 mg PO twice daily for 7 days to Mr. John Jones, a 64-year-old male with severe knee pain an"


## Quick Entity Type Counts

In [41]:
if all_entities.empty:
    entity_counts = pd.DataFrame(columns=["method", "source", "entity", "count"])
else:
    entity_counts = (
        all_entities.groupby(["method", "source", "entity"])
        .size()
        .reset_index(name="count")
        .sort_values(["method", "source", "entity"])
        .reset_index(drop=True)
    )

entity_counts

,method,source,entity,count
0,regex,regex:ADDRESS,ADDRESS,1
1,regex,regex:AGE,AGE,1
2,regex,regex:BLOOD_PRESSURE,BLOOD_PRESSURE,1
3,regex,regex:CITY_STATE_ZIP,CITY_STATE_ZIP,1
4,regex,regex:CPT_CODE,CPT_CODE,1
5,regex,regex:DATE,DATE,1
6,regex,regex:DOCTOR_NAME,DOCTOR_NAME,1
7,regex,regex:DOSAGE,DOSAGE,2
8,regex,regex:DURATION,DURATION,1
9,regex,regex:EMAIL,EMAIL,1
